# Day 1 — Setup & Data
## Trustworthy Evidence-Grounded Framework · FEVEROUS Validation

**Objectives:**
1. Install all required packages
2. Mount Google Drive (for persistence across sessions)
3. Load FEVEROUS from Hugging Face
4. Build a balanced 450-claim sample (150 per label)
5. Resolve gold evidence text (sentences + linearized table cells)
6. Add ~2,000 distractor evidence items for non-trivial retrieval
7. Save frozen `claims.json` and `evidence_corpus.json`
8. Record and print dataset statistics for Section 4.2 of the paper

---
**Expected runtime:** ~15-25 min on Colab free tier (dataset download + SQLite resolution)

In [ ]:
# Cell 1: Install packages
!pip install -q datasets sentence-transformers faiss-cpu transformers accelerate bitsandbytes huggingface_hub
print('Packages installed')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/feverous-validation'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUT_DIR  = os.path.join(BASE_DIR, 'outputs')
FIG_DIR  = os.path.join(BASE_DIR, 'figures')

for d in [DATA_DIR, OUT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {BASE_DIR}')

In [ ]:
# Cell 3: Imports
import json
import random
import sqlite3
import re
import urllib.request
import zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
from datasets import load_dataset

random.seed(42)
np.random.seed(42)
print('Imports ready')

In [ ]:
# Cell 4: Load FEVEROUS dataset from Hugging Face
print('Loading fever/feverous from Hugging Face ...')
ds = load_dataset('fever/feverous', trust_remote_code=True)
print(ds)
print('Column names:', ds['train'].column_names)

In [ ]:
# Cell 5: Inspect a sample row to understand field names
sample = ds['train'][0]
for k, v in sample.items():
    print(f'{k}: {str(v)[:200]}')

In [ ]:
# Cell 6: Filter and balance to 150 claims per label
VALID_LABELS = {'SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO'}
TARGET_PER_LABEL = 150

buckets = defaultdict(list)
for row in ds['train']:
    label = row.get('label', '').upper().strip()
    if label in VALID_LABELS and len(buckets[label]) < TARGET_PER_LABEL:
        buckets[label].append(row)

sampled_rows = []
for label in sorted(VALID_LABELS):
    print(f'  {label}: {len(buckets[label])} claims')
    sampled_rows.extend(buckets[label])

random.shuffle(sampled_rows)
print(f'Total sampled claims: {len(sampled_rows)}')

In [ ]:
# Cell 7: Download FEVEROUS Wikipedia SQLite DB
# The DB is needed to resolve table-cell evidence IDs to text.
# Download from fever.ai (~7.5 GB). This may take 10-20 min on free Colab.

SQLITE_URL  = 'https://fever.ai/download/feverous/feverous-wiki-pages-db.zip'
SQLITE_PATH = '/content/feverous-wiki-pages-db.db'
SQLITE_ZIP  = '/content/feverous-wiki-pages.zip'

if not os.path.exists(SQLITE_PATH):
    print('Downloading FEVEROUS Wikipedia SQLite DB (~7.5 GB) ...')
    print('This may take 10-20 min. Grab a coffee!')
    urllib.request.urlretrieve(SQLITE_URL, SQLITE_ZIP)
    print('Extracting ...')
    with zipfile.ZipFile(SQLITE_ZIP, 'r') as z:
        z.extractall('/content/')
    # Rename extracted file if needed
    extracted_dbs = [f for f in os.listdir('/content/') if f.endswith('.db')]
    if extracted_dbs and extracted_dbs[0] != 'feverous-wiki-pages-db.db':
        os.rename(f'/content/{extracted_dbs[0]}', SQLITE_PATH)
    print('SQLite DB ready:', SQLITE_PATH)
else:
    print('SQLite DB already present')

In [ ]:
# Cell 8: Evidence resolution helpers

def get_db_conn(db_path=SQLITE_PATH):
    conn = sqlite3.connect(db_path)
    return conn


def linearize_table_cell(page_data, table_id, row_id, cell_id):
    tables = page_data.get('tables', [])
    if table_id >= len(tables):
        return f'[TABLE {table_id} NOT FOUND]'
    table = tables[table_id]
    rows = table.get('rows', table if isinstance(table, list) else [])
    col_names = []
    if rows:
        header_cells = rows[0] if isinstance(rows[0], list) else []
        col_names = [str(c.get('value', c) if isinstance(c, dict) else c) for c in header_cells]
    if row_id >= len(rows):
        return f'[ROW {row_id} NOT FOUND]'
    row = rows[row_id]
    cells = row if isinstance(row, list) else []
    if cell_id >= len(cells):
        return f'[CELL {cell_id} NOT FOUND]'
    cell = cells[cell_id]
    val = str(cell.get('value', cell) if isinstance(cell, dict) else cell).strip()
    col = col_names[cell_id] if cell_id < len(col_names) else f'Col{cell_id}'
    return f'Table | Row {row_id} | {col}: {val}'


def linearize_full_table(page_data, table_id):
    tables = page_data.get('tables', [])
    if table_id >= len(tables):
        return None
    table = tables[table_id]
    rows = table.get('rows', table if isinstance(table, list) else [])
    parts = []
    header = None
    for i, row in enumerate(rows[:10]):
        cells = row if isinstance(row, list) else []
        vals = [str(c.get('value', c) if isinstance(c, dict) else c).strip() for c in cells]
        if i == 0:
            header = vals
        else:
            if header:
                parts.append(', '.join(f'{h}: {v}' for h, v in zip(header, vals)))
            else:
                parts.append(', '.join(vals))
    return ' | '.join(parts) if parts else None


_page_cache = {}

def get_page_data(page_title, conn):
    if page_title in _page_cache:
        return _page_cache[page_title]
    cur = conn.cursor()
    cur.execute('SELECT data FROM wiki WHERE id = ?', (page_title,))
    row = cur.fetchone()
    if row is None:
        _page_cache[page_title] = None
        return None
    try:
        data = json.loads(row[0])
    except Exception:
        data = None
    _page_cache[page_title] = data
    return data


def parse_evidence_id(ev_id):
    m = re.match(r'^(.+)_cell_(\d+)_(\d+)_(\d+)$', ev_id)
    if m:
        return {'page': m.group(1), 'type': 'cell',
                'table_id': int(m.group(2)),
                'row_id':   int(m.group(3)),
                'col_id':   int(m.group(4))}
    m = re.match(r'^(.+)_sentence_(\d+)$', ev_id)
    if m:
        return {'page': m.group(1), 'type': 'sentence',
                'sent_id': int(m.group(2))}
    return None


def resolve_evidence_id(ev_id, conn):
    parsed = parse_evidence_id(ev_id)
    if parsed is None:
        return f'[UNPARSEABLE ID: {ev_id}]', 'unknown'
    page_data = get_page_data(parsed['page'], conn)
    if page_data is None:
        return f'[PAGE NOT FOUND: {parsed["page"]}]', parsed['type']
    if parsed['type'] == 'sentence':
        sents = page_data.get('sentences', [])
        sid = parsed['sent_id']
        if sid < len(sents):
            s = sents[sid]
            text = s.get('content', str(s)) if isinstance(s, dict) else str(s)
            return text.strip(), 'sentence'
        return f'[SENTENCE {sid} NOT FOUND]', 'sentence'
    else:
        text = linearize_table_cell(
            page_data, parsed['table_id'], parsed['row_id'], parsed['col_id']
        )
        return text, 'cell'


print('Evidence resolution helpers defined')

In [ ]:
# Cell 9: Resolve gold evidence for all sampled claims
conn = get_db_conn()

claims_out   = []
gold_ev_texts = []
n_missing = 0

for i, row in enumerate(sampled_rows):
    claim_id   = row.get('id', i)
    claim_text = row.get('claim', '').strip()
    label      = row.get('label', '').upper().strip()

    evidence_sets = row.get('evidence', [[]])
    all_ev_ids = []
    for ev_set in evidence_sets:
        if isinstance(ev_set, list):
            all_ev_ids.extend(ev_set)
        elif isinstance(ev_set, str):
            all_ev_ids.append(ev_set)

    resolved_evidence = []
    for ev_id in all_ev_ids:
        if not ev_id or not isinstance(ev_id, str):
            continue
        text, ev_type = resolve_evidence_id(ev_id, conn)
        if 'NOT FOUND' in text or 'UNPARSEABLE' in text:
            n_missing += 1
            continue
        resolved_evidence.append({'ev_id': ev_id, 'text': text, 'type': ev_type})
        gold_ev_texts.append({'ev_id': ev_id, 'text': text, 'type': ev_type,
                              'is_gold': True, 'claim_id': claim_id})

    claims_out.append({'claim_id': claim_id, 'claim': claim_text,
                       'label': label, 'gold_evidence': resolved_evidence})

    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(sampled_rows)} claims processed, {n_missing} unresolved IDs')

print(f'Resolved {len(claims_out)} claims')
print(f'Total gold evidence items: {len(gold_ev_texts)}')
print(f'Unresolvable evidence IDs: {n_missing}')

In [ ]:
# Cell 10: Build distractor corpus from unrelated Wikipedia content
DISTRACTOR_TARGET = 2500

gold_pages = set()
for ev in gold_ev_texts:
    parsed = parse_evidence_id(ev['ev_id'])
    if parsed:
        gold_pages.add(parsed['page'])

print(f'Gold pages (excluded from distractors): {len(gold_pages)}')

cur = conn.cursor()
cur.execute('SELECT id, data FROM wiki ORDER BY RANDOM() LIMIT 5000')

distractors = []
for page_title, raw_data in cur:
    if page_title in gold_pages:
        continue
    try:
        page_data = json.loads(raw_data)
    except Exception:
        continue

    sents = page_data.get('sentences', [])
    for j, s in enumerate(sents[:3]):
        text = s.get('content', str(s)) if isinstance(s, dict) else str(s)
        text = text.strip()
        if len(text) < 20:
            continue
        ev_id = f'{page_title}_sentence_{j}'
        distractors.append({'ev_id': ev_id, 'text': text, 'type': 'sentence',
                            'is_gold': False, 'claim_id': None})
        if len(distractors) >= DISTRACTOR_TARGET:
            break

    for t_id in range(len(page_data.get('tables', []))):
        lrow = linearize_full_table(page_data, t_id)
        if lrow and len(lrow) > 20:
            ev_id = f'{page_title}_table_{t_id}_linearized'
            distractors.append({'ev_id': ev_id, 'text': lrow, 'type': 'table_row',
                                'is_gold': False, 'claim_id': None})
        if len(distractors) >= DISTRACTOR_TARGET:
            break

    if len(distractors) >= DISTRACTOR_TARGET:
        break

print(f'Distractor items collected: {len(distractors)}')

In [ ]:
# Cell 11: Build and deduplicate the evidence corpus
seen_ids = set()
evidence_corpus = []

for ev in gold_ev_texts:
    if ev['ev_id'] not in seen_ids:
        seen_ids.add(ev['ev_id'])
        evidence_corpus.append(ev)

for ev in distractors:
    if ev['ev_id'] not in seen_ids and len(ev['text'].strip()) >= 15:
        seen_ids.add(ev['ev_id'])
        evidence_corpus.append(ev)

random.shuffle(evidence_corpus)

for idx, ev in enumerate(evidence_corpus):
    ev['corpus_idx'] = idx

ev_id_to_idx = {ev['ev_id']: ev['corpus_idx'] for ev in evidence_corpus}
for claim in claims_out:
    for gev in claim['gold_evidence']:
        gev['corpus_idx'] = ev_id_to_idx.get(gev['ev_id'], -1)

gold_count = sum(1 for ev in evidence_corpus if ev['is_gold'])
type_counts = Counter(ev['type'] for ev in evidence_corpus)

print(f'Evidence corpus size: {len(evidence_corpus)}')
print(f'  Gold items:       {gold_count}')
print(f'  Distractor items: {len(evidence_corpus)-gold_count}')
print(f'  Type breakdown:   {dict(type_counts)}')

In [ ]:
# Cell 12: Save frozen dataset
claims_path = os.path.join(DATA_DIR, 'claims.json')
corpus_path = os.path.join(DATA_DIR, 'evidence_corpus.json')

with open(claims_path, 'w', encoding='utf-8') as f:
    json.dump(claims_out, f, ensure_ascii=False, indent=2)

with open(corpus_path, 'w', encoding='utf-8') as f:
    json.dump(evidence_corpus, f, ensure_ascii=False, indent=2)

print(f'claims.json saved to {claims_path}')
print(f'evidence_corpus.json saved to {corpus_path}')

with open(claims_path) as f:
    chk = json.load(f)
print(f'Verification: {len(chk)} claims reloaded OK')

In [ ]:
# Cell 13: Print dataset statistics for paper Section 4.2
label_counts    = Counter(c['label'] for c in claims_out)
gold_per_claim  = [len(c['gold_evidence']) for c in claims_out]
type_mix_gold   = Counter(ev['type'] for c in claims_out for ev in c['gold_evidence'])

print('=' * 60)
print('DATASET STATISTICS  (Section 4.2)')
print('=' * 60)
print(f'Total claims: {len(claims_out)}')
for lbl, cnt in sorted(label_counts.items()):
    print(f'  {lbl:<24} {cnt}')
print()
print(f'Evidence corpus size:  {len(evidence_corpus)}')
print(f'  Gold evidence items: {gold_count}')
print(f'  Distractor items:    {len(evidence_corpus)-gold_count}')
print(f'  Type breakdown (gold): {dict(type_mix_gold)}')
print()
print(f'Gold evidence per claim:')
print(f'  Mean:   {np.mean(gold_per_claim):.2f}')
print(f'  Median: {np.median(gold_per_claim):.0f}')
print(f'  Max:    {np.max(gold_per_claim)}')
print(f'  Min:    {np.min(gold_per_claim)}')
print()
print('License: CC BY-SA 3.0 (Wikipedia via FEVEROUS)')
print('=' * 60)

stats = {
    'total_claims': len(claims_out),
    'label_counts': dict(label_counts),
    'corpus_size': len(evidence_corpus),
    'gold_count': gold_count,
    'distractor_count': len(evidence_corpus) - gold_count,
    'type_breakdown_gold': dict(type_mix_gold),
    'gold_per_claim': {
        'mean': round(float(np.mean(gold_per_claim)), 3),
        'median': float(np.median(gold_per_claim)),
        'max': int(np.max(gold_per_claim)),
        'min': int(np.min(gold_per_claim))
    }
}
with open(os.path.join(DATA_DIR, 'dataset_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)

print('dataset_stats.json saved')

## Day 1 Complete

Files saved to Google Drive:
- `data/claims.json` - 450 balanced claims with resolved gold evidence
- `data/evidence_corpus.json` - full evidence corpus (gold + distractors)
- `data/dataset_stats.json` - statistics for paper Section 4.2

Next: Open `Day2_Retrieval_Pipeline.ipynb`